In [26]:
import yaml
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cobra.io import read_sbml_model, write_sbml_model
from cobra.manipulation import remove_genes
from cobra.flux_analysis import flux_variability_analysis
from cobra.flux_analysis.parsimonious import optimize_minimal_flux

In [42]:
PRINT_DEMANDS = True
PLOT_BOUNDS = True

sbml_file = "../config/iML1515.xml"
model = read_sbml_model(sbml_file, f_replace={'F_REACTION': lambda x: x})
model.solver = "gurobi"
exchanges = [r.id for r in model.exchanges if r.id.startswith("R_EX")]

for r in model.reactions:
    if r.objective_coefficient != 0:
        objective_reaction = r.id
        break

print(f"Objective reaction: {objective_reaction}")

model.reactions.R_EX_o2_e.lower_bound = -50
model.reactions.R_EX_ac_e.lower_bound = -10
model.reactions.R_EX_lac__L_e.lower_bound = -10

solution = model.optimize()
print()
summary = None
if solution.status == 'optimal':
    summary = model.summary(solution)
    max_growth_rate = solution.objective_value
    print(f"Max growth rate: {max_growth_rate} 1/hr")
else:
    print("infeasible model")
summary

Objective reaction: R_BIOMASS_Ec_iML1515_core_75p37M

Max growth rate: 0.876997214426969 1/hr


Metabolite,Reaction,Flux,C-Number,C-Flux
M_ca2_e,R_EX_ca2_e,0.004565,0,0.00%
M_cl_e,R_EX_cl_e,0.004565,0,0.00%
M_cobalt2_e,R_EX_cobalt2_e,2.192E-05,0,0.00%
M_cu2_e,R_EX_cu2_e,0.0006218,0,0.00%
M_fe2_e,R_EX_fe2_e,8.072,0,0.00%
M_glc__D_e,R_EX_glc__D_e,10,6,100.00%
M_k_e,R_EX_k_e,0.1712,0,0.00%
M_mg2_e,R_EX_mg2_e,0.007608,0,0.00%
M_mn2_e,R_EX_mn2_e,0.000606,0,0.00%
M_mobd_e,R_EX_mobd_e,6.139E-06,0,0.00%


In [39]:
print("Running FVA...")
fva_df = flux_variability_analysis(model, processes=10, fraction_of_optimum=0.0)

mask_blocked = fva_df.abs().max(axis=1) < 1e-9

blocked_reactions = fva_df.index[mask_blocked].tolist()
gap_metabolites = [m.id for m in model.metabolites if len({r.id for r in m.reactions} - set(blocked_reactions)) == 0]
gap_genes = [g.id for g in model.genes if len({r.id for r in g.reactions} - set(blocked_reactions)) == 0]

Running FVA...


In [40]:
model_trimmed = model.copy()

print(" - Original model")
print(f"\t - Reactions {len(model.reactions)}")
print(f"\t - Metabolites {len(model.metabolites)}")
print(f"\t - Genes {len(model.genes)}")
print()

model_trimmed.remove_reactions(blocked_reactions)

gap_metabolites = [model_trimmed.metabolites.get_by_id(m) for m in gap_metabolites]
model_trimmed.remove_metabolites(gap_metabolites)

remove_genes(model_trimmed, gap_genes)

print(" - Trimmed model")
print(f"\t - Reactions {len(model_trimmed.reactions)}")
print(f"\t - Metabolites {len(model_trimmed.metabolites)}")
print(f"\t - Genes {len(model_trimmed.genes)}")

Read LP format model from file /tmp/tmpa80gmo_5.lp
Reading time = 0.01 seconds
: 1877 rows, 5424 columns, 21150 nonzeros
 - Original model
	 - Reactions 2712
	 - Metabolites 1877
	 - Genes 1516

 - Trimmed model
	 - Reactions 1744
	 - Metabolites 1147
	 - Genes 1106


In [41]:
sbml_out = "../config/iML1515_trimmed.xml"
write_sbml_model(model_trimmed, sbml_out, f_replace=None)